## References

Redshift Architecture - https://docs.aws.amazon.com/redshift/latest/dg/c_high_level_system_architecture.html

## Setup Redshift Multi-Node Cluster

Create a new cluster with 3 nodes.

Add ITVGitHubS3FullPolicy policy to the new cluster.

In the Multi Cluster Redshift Query Editor as awsuser:
```sql
CREATE DATABASE retail_db;

CREATE USER retail_user WITH PASSWORD 'Retail_P@ssw0rd';
-- in redshift, new user will have all permissions on all databases in the cluster 
```

In a new session, connect as retail_user to retail_db
```sql
CREATE TABLE orders (
    order_id INT,
    order_date TIMESTAMP,
    order_customer_id INT,
    order_status VARCHAR(30)
);
-- table should be created successfully in the public schema by default

DROP TABLE orders;
```

## Create Schemas

Connect to retail_db using awsuser
```sql
CREATE SCHEMA retail_ods AUTHORIZATION retail_user;

```

Change connection as retail_user to retail_db to validate 

retail_ods should be available under the Select Schema drop down.
```sql
-- create orders table under retail_ods schema, you could also choose schema in drop down menu
CREATE TABLE retail_ods.orders (
    order_id INT,
    order_date TIMESTAMP,
    order_customer_id INT,
    order_status VARCHAR(30)
);

```

## Redshift Tables - Distribution Style

By default, Redshift uses 'auto' distribution style when none is provided.

To explicitly use auto distribution style (connect to retail_db as retail_user):
```sql
DROP TABLE retail_ods.orders; 

CREATE TABLE retail_ods.orders (
    order_id INT,
    order_date TIMESTAMP,
    order_customer_id INT,
    order_status VARCHAR(30)
) DISTSTYLE AUTO;
```
Check the metadata for orders table to verify the distribution style.

But first, metadata tables are not accessible by other users except the admin user by default.

Change connection as awsuser, then grant permissions to pg_catalog schema to retail_user:
```sql
GRANT SELECT ON ALL TABLES IN SCHEMA pg_catalog TO retail_user;

SELECT * FROM pg_table_def WHERE schemaname = 'retail_ods';
-- this will return empty because the search path is wrong (awsuser under public schema)

-- to show search path use
SHOW search_path

-- the search path will need to be updated at the cluster level, setting it in the query editor will not work
```
Go to the Redshift Cluster settings, Default Parameter Group, look for search_path variable.

To change the search_path, you will need to create a custom parameter group. Use the name retail-multi-custom.

Change the search_path to: "$user, public, retail_ods".

Map the new parameter group to the Cluster.

Finally, reboot the Redshift Cluster for the changes to take effect.

When reboot is finished, connect as retail_user in retail_db and verify the changes:
```sql
SHOW search_path

SELECT * FROM pg_table_def WHERE schemaname = 'retail_ods';

SELECT * FROM svv_table_info;
-- will not return anything because orders table has no data yet

-- copy data from s3 into orders table
COPY retail_ods.orders
FROM 's3://romadv-itv-retail/retail_db_json/orders/'
IAM_ROLE 'arn of s3 full access role'
JSON AS 'auto';

SELECT * FROM retail_ods.orders LIMIT 10;

SELECT * FROM svv_table_info;
-- information should be available with diststyle = AUTO(ALL) for orders table
```
#### Distribution Strategies

We have 6 tables as part of retail database. Out of those 6 tables, 3 are master (dimension) tables for product catalog: departments, categories, and products. As these tables are small we can use ALL as DISTSTYLE for them. Customers is alo a master table however, it can grow larger over time. For now we can consider AUTO for it. Orders and order_items are transactional (fact) tables. Order_items is a child table to order. We typically join these 2 tables, hence we can consider a common key as DISTKEY. We can create orders table with order_id as distkey. In similar lines, we can create order_items table with order_item_order_id as distkey. It's the 2nd column in the table and is a foreign key to order.order_id.

Analyze the access patterns, join styles, and data distribution to determine the appropriate distribution strategy to apply to a table.

#### Create table with ALL distribution style
Connect to query editor as retail_user in retail_db
```sql
CREATE TABLE retail_ods.departments (
    department_id INT NOT NULL,
    department_name VARCHAR(45) NOT NULL,
    PRIMARY KEY (department_id)
) DISTSTYLE ALL;

COPY retail_ods.departments
FROM 's3://romadv-itv-retail/retail_db_json/departments/'
IAM_ROLE 'arn of s3 full access role'
JSON AS 'auto';

-- validation
SELECT * FROM retail_ods.departments LIMIT 10;

SELECT * FROM pg_table_def WHERE schemaname = 'retail_ods' AND tablename = 'departments';
-- distkey false as no distribution key is used

SELECT * FROM svv_table_info WHERE "table" = 'departments';
-- diststyle should show as 'ALL'

CREATE TABLE retail_ods.categories (
    category_id INT NOT NULL,
    category_department_id INT NOT NULL,
    category_name VARCHAR(45) NOT NULL,
    PRIMARY KEY (category_id)
) DISTSTYLE ALL;

COPY retail_ods.categories
FROM 's3://romadv-itv-retail/retail_db_json/categories/'
IAM_ROLE 'arn of s3 full access role'
JSON AS 'auto';

SELECT * FROM retail_ods.categories LIMIT 10;


CREATE TABLE retail_ods.products (
    product_id INT NOT NULL,
    product_category INT NOT NULL,
    product_name VARCHAR(45) NOT NULL,
    product_description VARCHAR(255) NOT NULL,
    product_price FLOAT NOT NULL,
    product_image VARCHAR(255) NOT NULL,
    PRIMARY KEY (product_id)
) DISTSTYLE ALL;

COPY retail_ods.products
FROM 's3://romadv-itv-retail/retail_db_json/products/'
IAM_ROLE 'arn of s3 full access role'
JSON AS 'auto';
-- will fail: string length exceeds DDL length for product name

SELECT * FROM stl_load_errors;

DROP TABLE retail_ods.products;
CREATE TABLE retail_ods.products (
    product_id INT NOT NULL,
    product_category INT NOT NULL,
    product_name VARCHAR(60) NOT NULL,
    product_description VARCHAR(255) NOT NULL,
    product_price FLOAT NOT NULL,
    product_image VARCHAR(255) NOT NULL,
    PRIMARY KEY (product_id)
) DISTSTYLE ALL;

SELECT * FROM retail_ods.products LIMIT 10;
```

#### Create customers table with disttyle AUTO
```sql
CREATE TABLE retail_ods.customers (
    customer_id INT NOT NULL,
    customer_fname VARCHAR(45) NOT NULL,
    customer_lname VARCHAR(45) NOT NULL
    customer_email VARCHAR(45) NOT NULL
    customer_password VARCHAR(45) NOT NULL
    customer_street VARCHAR(255) NOT NULL
    customer_city VARCHAR(45) NOT NULL
    customer_state VARCHAR(45) NOT NULL
    customer_zipcode VARCHAR(45) NOT NULL
    PRIMARY KEY (customer_id)
) DISTSTYLE AUTO;

COPY retail_ods.customers
FROM 's3://romadv-itv-retail/retail_db_json/customers/'
IAM_ROLE 'arn of s3 full access role'
JSON AS 'auto';

SELECT * FROM retail_ods.customers LIMIT 10;

SELECT * FROM pg_table_def WHERE schemaname = 'retail_ods' AND tablename = 'customers';

SELECT * FROM svv_table_info WHERE "table" = 'customers';
-- redshift will decide which diststyle to use which would most likely be AUTO(EVEN)
```
#### Create tables with DISTRIBUTION KEY used
```sql
DROP TABLE retail_ods.orders; 
CREATE TABLE retail_ods.orders (
    order_id INT NOT NULL DISTKEY,
    order_date TIMESTAMP NOT NULL,
    order_customer_id INT NOT NULL,
    order_status VARCHAR(45) NOT NULL,
    PRIMARY KEY (order_id)
) DISTSTYLE KEY;

COPY retail_ods.orders
FROM 's3://romadv-itv-retail/retail_db_json/orders/'
IAM_ROLE 'arn of s3 full access role'
JSON AS 'auto';

SELECT * FROM retail_ods.orders LIMIT 10;

SELECT * FROM pg_table_def WHERE schemaname = 'retail_ods' AND tablename = 'orders';
-- order_id should be the distkey, only 1 column can be distkey

SELECT * FROM svv_table_info WHERE "table" = 'customers';
-- diststyle = KEY(order_id)

CREATE TABLE retail_ods.order_items (
    order_item_id INT NOT NULL,
    order_item_order_id INT NOT NULL DISTKEY,
    order_item_product_id INT NOT NULL,
    order_item_quantity INT NOT NULL,
    order_item_subtotal FLOAT NOT NULL,
    order_item_product_price FLOAT NOT NULL,
    PRIMARY KEY (order_item_id),
    FOREIGN KEY (order_item_order_id) REFERENCES orders (order_id)
);
-- note: no need to specify distribution style DISTKEY column is specified

COPY retail_ods.order_items
FROM 's3://romadv-itv-retail/retail_db_json/order_items/'
IAM_ROLE 'arn of s3 full access role'
JSON AS 'auto';

SELECT * FROM retail_ods.order_items LIMIT 10;

SELECT * FROM pg_table_def WHERE schemaname = 'retail_ods' AND tablename = 'order_items';

SELECT * FROM svv_table_info WHERE "table" = 'order_items';
```